<a href="https://colab.research.google.com/github/DJ-Ortega/unsupervised-trading-strat/blob/main/Unsupervised_strat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unsupervised Learning Trading Strategy

* Download/Load SP500 stocks prices data.
* Calculate different features and indicators on each stock.
* Aggregate on monthly level and filter top 150 most liquid stocks.
* Calculate Monthly Returns for different time-horizons.
* Download Fama-French Factors and Calculate Rolling Factor Betas.
* For each month fit a K-Means Clustering Algorithm to group similar assets based on their features.
* For each month select assets based on the cluster and form a portfolio based on Efficient Frontier max sharpe ratio optimization.
* Visualize Portfolio returns and compare to SP500 returns.

Be mindful that this backtest does not take in survivorship bias

In [ ]:
!pip install pandas-datareader --quiet
!pip install pyportfolioopt --quiet
!pip install pandas_ta --quiet
!pip install --upgrade yfinance --quiet
!pip install pandas==2.3.3 --quiet
!pip install numpy==2.2.6 --quiet

In [ ]:
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta
import warnings
warnings.filterwarnings('ignore')

## 1. Download/Load SP500 stocks prices data.

In [ ]:
import requests
from io import StringIO

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (student learning project)'}
response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))
sp500 = tables[0]
sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')
symbols_list = sp500['Symbol'].unique().tolist()

end_date = '2026-10-30'
start_date = pd.to_datetime(end_date) - pd.DateOffset(365*8)

raw_df = yf.download(tickers=symbols_list,
                     start=start_date,
                     end=end_date)

[*********************100%***********************]  503 of 503 completed


In [ ]:
#cleaning the dataframe
df = raw_df.stack()
df.index.names = ['date', 'ticker']
df.columns = df.columns.str.lower()
df['adj close'] = df['close']
df

Price                   close        high         low        open  \
date       ticker                                                   
2018-11-01 A        61.491661   61.604797   60.888248   61.350234   
           AAPL     52.496876   52.529949   51.218826   51.748001   
           ABBV     57.475544   57.790834   55.533658   55.684136   
           ABT      60.562431   60.735094   59.345136   59.647300   
           ACGL     27.281279   27.576057   26.739267   27.119626   
...                       ...         ...         ...         ...   
2026-10-29 XYZ      76.949997   77.559998   74.809998   74.830002   
           YUM     154.550003  154.750000  152.000000  152.520004   
           ZBH      97.470001   97.989998   96.260002   97.330002   
           ZBRA    377.380005  389.230011  374.549988  384.929993   
           ZTS      78.270004   78.746772   77.236998   77.912425   

Price                   volume   adj close  
date       ticker                           
2018-11-01 A         1882400.0   61.491661  
           AAPL    233292800.0   52.496876  
           ABBV     10671700.0   57.475544  
           ABT       6250800.0   60.562431  
           ACGL      2081000.0   27.281279  
...                        ...         ...  
2026-10-29 XYZ       4331600.0   76.949997  
           YUM       1606700.0  154.550003  
           ZBH       2040800.0   97.470001  
           ZBRA       527500.0  377.380005  
           ZTS       2935500.0   78.270004  

[988237 rows x 6 columns]

## 2. Calculate features and technical indicators for each stock.

* Garman-Klass Volatility
* RSI
* Bollinger Bands
* ATR
* MACD
* Dollar Volume

\begin{equation}
\text{Garman-Klass Volatility} = \frac{(\ln(\text{High}) - \ln(\text{Low}))^2}{2} - (2\ln(2) - 1)(\ln(\text{Adj Close}) - \ln(\text{Open}))^2
\end{equation}

$$z = \frac{\text{ATR} - \text{mean}}{\text{std}}$$

In [ ]:
df['garman_klass_vol'] = ((np.log(df['high'])-np.log(df['low']))**2)/2 - (2*np.log(2)-1)*(np.log(df['adj close'])-np.log(df['open']))**2

df['rsi'] = df.groupby(level = 1)['adj close'].transform(lambda x: pandas_ta.rsi(close = x, length = 20))
#df.xs('AAPL', level=1)['rsi'].plot()

df['bb_low'] = df.groupby(level = 1)['adj close'].transform(lambda x: pandas_ta.bbands(close = np.log1p(x), length = 20).iloc[:,0])

df['bb_mid'] = df.groupby(level = 1)['adj close'].transform(lambda x: pandas_ta.bbands(close = np.log1p(x), length = 20).iloc[:,1])

df['bb_high'] = df.groupby(level = 1)['adj close'].transform(lambda x: pandas_ta.bbands(close = np.log1p(x), length = 20).iloc[:,2])

# atr function
def compute_atr(stock_data):
  atr = pandas_ta.atr(high = stock_data['high'],
                      low = stock_data['low'],
                      close = stock_data['adj close'],
                      length = 14)
  return atr.sub(atr.mean()).div(atr.std())

df['atr'] = df.groupby(level = 1, group_keys = False).apply(compute_atr)

# macd function
def compute_macd(close):
    macd = pandas_ta.macd(close=close)
    if macd is None:
        return pd.Series(np.nan, index=close.index)
    macd = macd.iloc[:,0]
    return macd.sub(macd.mean()).div(macd.std())

df['macd'] = df.groupby(level = 1, group_keys = False)['adj close'].apply(compute_macd)

df['dollar_volume'] = (df['adj close']*df['volume'])/1e6 #divide by million

df


Price                   close        high         low        open  \
date       ticker                                                   
2018-11-01 A        61.491661   61.604797   60.888248   61.350234   
           AAPL     52.496876   52.529949   51.218826   51.748001   
           ABBV     57.475544   57.790834   55.533658   55.684136   
           ABT      60.562431   60.735094   59.345136   59.647300   
           ACGL     27.281279   27.576057   26.739267   27.119626   
...                       ...         ...         ...         ...   
2026-10-29 XYZ      76.949997   77.559998   74.809998   74.830002   
           YUM     154.550003  154.750000  152.000000  152.520004   
           ZBH      97.470001   97.989998   96.260002   97.330002   
           ZBRA    377.380005  389.230011  374.549988  384.929993   
           ZTS      78.270004   78.746772   77.236998   77.912425   

Price                   volume   adj close  garman_klass_vol        rsi  \
date       ticker                                                         
2018-11-01 A         1882400.0   61.491661          0.000066        NaN   
           AAPL    233292800.0   52.496876          0.000240        NaN   
           ABBV     10671700.0   57.475544          0.000406        NaN   
           ABT       6250800.0   60.562431          0.000178        NaN   
           ACGL      2081000.0   27.281279          0.000461        NaN   
...                        ...         ...               ...        ...   
2026-10-29 XYZ       4331600.0   76.949997          0.000350  49.542236   
           YUM       1606700.0  154.550003          0.000093  61.320330   
           ZBH       2040800.0   97.470001          0.000158  56.610909   
           ZBRA       527500.0  377.380005          0.000587  53.726565   
           ZTS       2935500.0   78.270004          0.000179  56.769603   

Price                bb_low    bb_mid   bb_high       atr      macd  \
date       ticker                                                     
2018-11-01 A            NaN       NaN       NaN       NaN       NaN   
           AAPL         NaN       NaN       NaN       NaN       NaN   
           ABBV         NaN       NaN       NaN       NaN       NaN   
           ABT          NaN       NaN       NaN       NaN       NaN   
           ACGL         NaN       NaN       NaN       NaN       NaN   
...                     ...       ...       ...       ...       ...   
2026-10-29 XYZ     4.314677  4.352530  4.390383 -0.763525 -0.078185   
           YUM     4.924020  4.997748  5.071475  1.515138  1.373804   
           ZBH     4.475066  4.549015  4.622964 -0.477666  0.560336   
           ZBRA    5.918707  5.947949  5.977191  0.369165  0.428429   
           ZTS     4.246431  4.344412  4.442392 -1.572758  0.631972   

Price              dollar_volume  
date       ticker                 
2018-11-01 A          115.751903  
           AAPL     12247.143138  
           ABBV       613.361763  
           ABT        378.563646  
           ACGL        56.772341  
...                          ...  
2026-10-29 XYZ        333.316607  
           YUM        248.315490  
           ZBH        198.916778  
           ZBRA       199.067953  
           ZTS        229.761598  

[988237 rows x 14 columns]